# HMP-GNN Colab — V1 + V2 M7 End-to-End Demo

This notebook reproduces the HMP-GNN paper (*Hallucination Immunization for Multimodal Federated LLMs via Hypergraph Message Passing*) pipeline in a few minutes on a free Colab GPU, and displays every paper-grade figure inline so you can save them with one click.

## What you will get

1. A clean federated-learning run (**No Attack**) as the reference.
2. A **Hallucination attack** (label-flipping, 2 malicious clients out of 10) aggregated with vanilla **FedAvg**.
3. The same attack aggregated with **HMP-GAE**, our server-side hypergraph-message-passing defense.
4. Four figures plus a numeric summary:
   - **Fig A** — final clean accuracy per defense.
   - **Fig C** — per-client trust-weight evolution (attackers collapse to ~0).
   - **Fig E** — three-panel bar: Accuracy ↑ / Classification Semantic Entropy ↓ / Perplexity ↓.
   - **Fig F** — per-round CSE evolution across the three runs.

## Before you run

1. Go to **Runtime → Change runtime type → T4 GPU** (the free tier is enough).
2. Run the cells top-to-bottom. The full demo takes **~10 minutes** on a T4.

## 1. Clone the repo and install dependencies

In [ ]:
# Clone the repository (skip if already cloned in this runtime).
import os
if not os.path.isdir('HMP-GNN'):
    !git clone https://github.com/GuangLun2000/HMP-GNN.git
%cd HMP-GNN
!pip install -q -r requirements.txt

In [ ]:
# Confirm GPU is available (optional but recommended).
import torch
print(f'PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Run the demo (3 experiments × ~3 min each)

`_v1_demo_run.py` runs the three comparable experiments back-to-back (N=10 clients, 3 rounds, DistilBERT + LoRA on an 800-sample AG News subset) and writes everything to `results/_v1_demo/`:

- **Run 1**: NoAttack → clean FL baseline.
- **Run 2**: Hallucination + FedAvg → confirms the attack hurts accuracy.
- **Run 3**: Hallucination + HMP-GAE → confirms our defense recovers accuracy and lowers semantic entropy.

By default PPL is *skipped* for encoder-only backbones like DistilBERT (see the PPL panel of Fig E). To get real PPL numbers, switch `model_name` to `Qwen/Qwen2.5-0.5B-Instruct` or `EleutherAI/pythia-160m` in the custom-run cell further down.

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
!python _v1_demo_run.py

## 3. Display all key figures inline

After the demo finishes, `results/_v1_demo/` contains four figures (each as `.png` + `.pdf`). The cell below displays each inline so you can right-click → **Save image** to grab them, or use the download cell at the end to zip them up.

In [ ]:
from IPython.display import Image, display, Markdown
from pathlib import Path

FIG_DIR = Path('results/_v1_demo')
KEY_FIGS = [
    ('Fig A — Defense Accuracy Bar',
     'figA_defense_bar.png',
     'Main accuracy comparison: HMP-GAE vs FedAvg under Hallucination attack.'),
    ('Fig C — Trust-Weight Evolution',
     'figC_trust_evolution.png',
     'Per-client alpha over rounds. Attackers (red) collapse to ~0; benign clients (blue) share mass uniformly.'),
    ('Fig E — Three-metric Grouped Bar (V2 M7)',
     'figE_metrics_grouped.png',
     'Accuracy (higher better) / CSE (lower better) / PPL (lower better). PPL panel shows "unavailable" for encoder-only backbones.'),
    ('Fig F — CSE Evolution (V2 M7)',
     'figF_cse_evolution.png',
     'Per-round Classification Semantic Entropy. HMP-GAE should track the NoAttack curve more closely than FedAvg.'),
]

for title, fname, caption in KEY_FIGS:
    path = FIG_DIR / fname
    if path.is_file():
        display(Markdown(f'### {title}\n{caption}\n\n`{path}`'))
        display(Image(filename=str(path)))
    else:
        display(Markdown(f'### {title}\n**Missing:** `{path}` (did the demo cell finish?)'))

### Numeric summary

This cell reads the three JSONs written by the demo and prints the Accuracy / CSE / PPL triplet per run. Useful for dropping straight into a paper table or Slack.

In [ ]:
from visualization import summarize_run_multi_metric
from pathlib import Path

runs = [
    ('No Attack',       'v1demo_noattack'),
    ('Hallu + FedAvg',  'v1demo_hallu_fedavg'),
    ('Hallu + HMP-GAE', 'v1demo_hallu_hmpgae'),
]
print(f"{'Run':20s}  {'Acc':>8s}  {'mean CSE':>10s}  {'PPL':>12s}")
print('-' * 54)
for label, stem in runs:
    res = Path('results') / f'{stem}_results.json'
    ppl = Path('results') / f'{stem}_eval_ppl.json'
    if not res.is_file():
        print(f'{label:20s}  (missing {res})')
        continue
    s = summarize_run_multi_metric(res, ppl_json_path=ppl if ppl.is_file() else None)
    acc = s.get('final_clean_acc', 0.0)
    cse = s.get('mean_cse')
    p   = s.get('ppl')
    cse_s = f'{cse:.4f}' if cse is not None else 'N/A'
    p_s   = f'{p:.2f}'  if p   is not None else 'N/A (enc-only)'
    print(f'{label:20s}  {acc:8.4f}  {cse_s:>10s}  {p_s:>12s}')

## 4. Download all figures + JSON results as a single zip

Runs on Colab only (uses `google.colab.files.download`). If running locally, the zip is left at `results/hmp_gae_colab_bundle.zip` for you to grab.

In [ ]:
import shutil
from pathlib import Path

bundle_dir = Path('results/hmp_gae_colab_bundle')
bundle_dir.mkdir(parents=True, exist_ok=True)

# Curate: 4 key figures + 3 results JSONs + 3 PPL JSONs + seed summary if any.
to_bundle = []
to_bundle += list(Path('results/_v1_demo').glob('*'))
for stem in ['v1demo_noattack', 'v1demo_hallu_fedavg', 'v1demo_hallu_hmpgae']:
    for suffix in ['_results.json', '_eval_ppl.json']:
        p = Path('results') / f'{stem}{suffix}'
        if p.is_file():
            to_bundle.append(p)
for p in to_bundle:
    if p.is_file():
        shutil.copy2(p, bundle_dir / p.name)

zip_path = shutil.make_archive(str(bundle_dir), 'zip', root_dir=bundle_dir)
print(f'Bundle ready: {zip_path}')

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print(f'(Not on Colab -- the zip is at {zip_path}. Reason: {type(e).__name__})')

## 5. (Optional) Run your own customized experiment

Override any config field for a one-off run without touching `main.py`. Useful knobs:

- `num_clients`, `num_attackers`, `num_rounds`, `dataset_size_limit`
- `attack_method`: `'NoAttack'` | `'Hallucination'` | `'SignFlipping'` | `'Gaussian'` | `'ALIE'`
- `defense_method`: `'fedavg'` | `'hmp_gae'`
- `model_name`: `'distilbert-base-uncased'` (fast; PPL skipped) or `'EleutherAI/pythia-160m'` / `'Qwen/Qwen2.5-0.5B-Instruct'` (decoder, PPL measured)

In [ ]:
import main

config_overrides = {
    'experiment_name': 'colab_custom_run',
    'seed': 42,
    # FL setup
    'num_clients': 10,
    'num_attackers': 2,
    'num_rounds': 5,
    # Training
    'client_lr': 5e-5,
    'batch_size': 64,
    'test_batch_size': 128,
    'local_epochs': 1,
    'alpha': 0.0,
    # Data + model (swap model_name for Qwen or Pythia to get a real PPL number)
    'dataset': 'ag_news',
    'num_labels': 4,
    'max_length': 128,
    'data_distribution': 'non-iid',
    'dirichlet_alpha': 0.3,
    'dataset_size_limit': 4000,
    'use_lora': True,
    'lora_r': 8,
    'lora_alpha': 16,
    'model_name': 'distilbert-base-uncased',
    # Attack
    'attack_method': 'Hallucination',
    'hallu_flip_ratio': 1.0,
    'hallu_flip_mode': 'pairwise',
    'hallu_flip_map': {0: 1, 1: 0, 2: 3, 3: 2},
    # Defense (flip to 'fedavg' to run the baseline)
    'defense_method': 'hmp_gae',
    # V2 M7 metrics (CSE every round, PPL at end-of-FL).
    'eval_classification_semantic_entropy': True,
    'eval_perplexity': True,
    'ppl_num_samples': 200,
    'ppl_seed': 42,
    # Keep downstream Task 2 off for the quick demo.
    'save_global_checkpoint': True,
    'run_downstream_after_fl': False,
}

main.main(config_overrides=config_overrides)

## 6. Next steps (V2 roadmap)

V1 + V2 M7 here prove the pipeline works and produces the paper's three headline metrics. Upcoming V2 work:

- Comparison **baselines** (Krum / Median / Trimmed-Mean / FLTrust / FLDetector / Safe-FedLLM).
- Full-scale runs with Qwen2.5-0.5B on AG News / IMDB / DBpedia with 3+ seeds (for real PPL numbers).
- Ablations on HMP-GAE components (remove hypergraph → pairwise graph, remove historical consistency, etc.).

See the `## HMP-GAE Immunization (V1)` and `### V2 M7: Hallucination Evaluation Metrics` sections of the project README for details and current result numbers.